# EDA — RuFPBench DataCollectionAgent (Assignment 1)

This notebook performs basic EDA required by the course:

- distribution of classes (`label`)
- distribution of text lengths
- top-20 tokens

> Run `DataCollectionAgent` first to create `data/raw/merged_raw.parquet`,  
> or let the notebook run collection automatically.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/merged_raw.parquet")

if DATA_PATH.exists():
    df = pd.read_parquet(DATA_PATH)
else:
    from agents.data_collection_agent import DataCollectionAgent
    df = DataCollectionAgent("config.yaml").run()

df.head(), df.shape


In [ ]:
# Basic overview
display(df.sample(5, random_state=42))

print("Rows:", len(df))
print("Sources:")
display(df["source"].value_counts().to_frame("count"))

print("Labels:")
display(df["label"].value_counts(dropna=False).to_frame("count"))


In [ ]:
# Text length statistics
df = df.copy()
df["text_len_chars"] = df["text"].astype(str).str.len()
df["text_len_words"] = df["text"].astype(str).str.split().apply(len)

display(df[["text_len_chars", "text_len_words"]].describe())


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
df["text_len_chars"].hist(bins=50)
plt.title("Text length (chars)")
plt.xlabel("chars")
plt.ylabel("count")
plt.show()

plt.figure()
df["text_len_words"].hist(bins=50)
plt.title("Text length (words)")
plt.xlabel("words")
plt.ylabel("count")
plt.show()


In [ ]:
# Top-20 tokens (very simple; no stopword removal by default)
from sklearn.feature_extraction.text import CountVectorizer

def top_tokens(texts, n=20, max_features=5000):
    vec = CountVectorizer(max_features=max_features)
    X = vec.fit_transform(texts)
    counts = X.sum(axis=0).A1
    vocab = vec.get_feature_names_out()
    top_idx = counts.argsort()[::-1][:n]
    return pd.DataFrame({"token": vocab[top_idx], "count": counts[top_idx]})

# overall
overall = top_tokens(df["text"].astype(str).tolist())
display(overall)

# per label (if not too many labels)
for lab, sub in df.groupby("label"):
    if len(sub) < 50:
        continue
    print(f"\nLabel={lab} (n={len(sub)})")
    display(top_tokens(sub["text"].astype(str).tolist()))
